# Building Permit Cost Predictor - Exploratory Data Analysis

This notebook explores the Calgary Building Permits dataset to understand patterns,
distributions, and relationships that will inform our ML model.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Load Data

In [ ]:
df_raw = load_or_fetch_data('../data', limit=100000)
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

## 2. Data Quality Assessment

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=False)

## 3. Preprocess & Engineer Features

In [ ]:
df = preprocess_data(df_raw)
df = engineer_features(df)
print(f'Processed dataset shape: {df.shape}')
df.head()

## 4. Target Variable Analysis

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Original Cost Distribution', 'Log-Transformed Cost'])
fig.add_trace(go.Histogram(x=df['estprojectcost'], nbinsx=50, marker_color='#667eea'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['log_cost'], nbinsx=50, marker_color='#764ba2'), row=1, col=2)
fig.update_layout(height=400, showlegend=False, title='Estimated Project Cost Distribution')
fig.show()

In [ ]:
print('Cost Statistics:')
print(df['estprojectcost'].describe())
print(f"\nSkewness: {df['estprojectcost'].skew():.2f}")
print(f"Kurtosis: {df['estprojectcost'].kurtosis():.2f}")
print(f"\nLog Cost Skewness: {df['log_cost'].skew():.2f}")
print(f"Log Cost Kurtosis: {df['log_cost'].kurtosis():.2f}")

## 5. Categorical Feature Analysis

In [ ]:
for col in ['permittype', 'permitclassgroup', 'workclassgroup']:
    if col in df.columns:
        fig = px.box(df, x=col, y='estprojectcost', color=col,
                     title=f'Cost Distribution by {col}')
        fig.update_layout(showlegend=False, height=400)
        fig.show()

## 6. Temporal Trends

In [ ]:
if 'apply_year' in df.columns:
    yearly = df.groupby('apply_year').agg(
        count=('estprojectcost', 'count'),
        mean_cost=('estprojectcost', 'mean'),
        median_cost=('estprojectcost', 'median'),
        total_value=('estprojectcost', 'sum')
    ).reset_index()

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(x=yearly['apply_year'], y=yearly['count'],
                         name='Permits', marker_color='#667eea', opacity=0.7), secondary_y=False)
    fig.add_trace(go.Scatter(x=yearly['apply_year'], y=yearly['median_cost'],
                            name='Median Cost', line=dict(color='#ff6b6b', width=3)), secondary_y=True)
    fig.update_layout(title='Annual Permit Volume & Median Cost', height=450)
    fig.update_yaxes(title_text='Permit Count', secondary_y=False)
    fig.update_yaxes(title_text='Median Cost ($)', secondary_y=True)
    fig.show()

## 7. Community Analysis

In [ ]:
if 'communityname' in df.columns:
    top_communities = df['communityname'].value_counts().head(20)
    fig = px.bar(x=top_communities.index, y=top_communities.values,
                 title='Top 20 Communities by Permit Count',
                 labels={'x': 'Community', 'y': 'Permit Count'})
    fig.update_layout(xaxis_tickangle=-45, height=450)
    fig.show()

## 8. Correlation Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_with_cost = df[numeric_cols].corr()['log_cost'].sort_values(ascending=False)
print('Correlation with Log Cost:')
print(corr_with_cost)

In [ ]:
# Correlation heatmap
key_numeric = [c for c in ['log_cost', 'totalsqft', 'housingunits', 'apply_year',
               'community_avg_cost', 'community_median_cost', 'latitude', 'longitude']
               if c in df.columns]
fig = px.imshow(df[key_numeric].corr(), text_auto='.2f',
                title='Feature Correlation Heatmap', color_continuous_scale='RdBu_r')
fig.update_layout(height=500)
fig.show()

## 9. Key Takeaways

1. **Cost distribution is highly right-skewed** - log transformation is essential
2. **Permit class is the strongest categorical predictor** - commercial permits cost significantly more
3. **Square footage correlates with cost** but has many missing values
4. **Community-level features** (average cost, median cost) are strong predictors
5. **Temporal trends** show rising costs over time with seasonal patterns
6. **Geographic location** (lat/lon) captures some community-level variation